# GhostPlayer: Process NFL Big Data Bowl 2026 Data

This notebook processes the 2026 Big Data Bowl Analytics files already placed under `data/`. It converts the competition's `input_*.csv` and `output_*.csv` files into GhostPlayer graph datasets under `processed/`.

Schema note: the 2026 files provide focused player subsets, not all 22 players plus the ball. This notebook pads each play to 23 graph nodes, uses the final node as a ball-landing context node, and predicts the full future trajectory for players marked `player_to_predict`.

In [1]:
from pathlib import Path
import json


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "PLAN.md").exists():
            return candidate
    raise RuntimeError("Could not find the GhostPlayer project root.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK_FRAMES = 10
TARGET_OUTPUT_FRAME_ID = 1
TRAJECTORY_HORIZON = None  # None keeps the full future horizon available in output files.

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: /Users/reezhan/Documents/GhostPlayer
Data dir: /Users/reezhan/Documents/GhostPlayer/data
Processed dir: /Users/reezhan/Documents/GhostPlayer/processed


## Load 2026 Tables

In [2]:
from ghostplayer.data.bdb2026 import find_2026_data_root, load_2026_tables


data_root = find_2026_data_root(DATA_DIR)
tables = load_2026_tables(data_root)

print(f"2026 data root: {data_root}")
print(f"supplementary: {tables.supplementary.shape}")
print(f"input_tracking: {tables.input_tracking.shape}")
print(f"output_tracking: {tables.output_tracking.shape}")
print(f"weeks: {sorted(tables.input_tracking['source_week'].dropna().unique().tolist())}")

2026 data root: /Users/reezhan/Documents/GhostPlayer/data/114239_nfl_competition_files_published_analytics_final
supplementary: (18009, 41)
input_tracking: (4880579, 25)
output_tracking: (562936, 8)
weeks: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]


## Inspect Schema

In [3]:
display(tables.supplementary.head())
display(tables.input_tracking.head())
display(tables.output_tracking.head())

print("players per play")
display(tables.input_tracking.groupby(["game_id", "play_id"])["nfl_id"].nunique().describe())

print("players to predict per play")
display(
    tables.input_tracking[tables.input_tracking["player_to_predict"]]
    .groupby(["game_id", "play_id"])["nfl_id"]
    .nunique()
    .describe()
)

print("input frames per play")
display(tables.input_tracking.groupby(["game_id", "play_id"])["frame_id"].nunique().describe())

print("output frames per play")
display(tables.output_tracking.groupby(["game_id", "play_id"])["frame_id"].nunique().describe())

,game_id,season,week,game_date,game_time_eastern,home_team_abbr,visitor_team_abbr,play_id,play_description,quarter,...,team_coverage_type,penalty_yards,pre_penalty_yards_gained,yards_gained,expected_points,expected_points_added,pre_snap_home_team_win_probability,pre_snap_visitor_team_win_probability,home_team_win_probability_added,visitor_team_win_probility_added
0,2023090700,2023,1,09/07/2023,20:20:00,KC,DET,3461,(10:46) (Shotgun) J.Goff pass deep left to J.R...,4,...,COVER_2_ZONE,NaN,18,18,-0.664416,2.945847,0.834296,0.165704,-0.081149,0.081149
1,2023090700,2023,1,09/07/2023,20:20:00,KC,DET,461,(7:30) J.Goff pass short right to J.Reynolds t...,1,...,COVER_6_ZONE,NaN,21,21,1.926131,1.345633,0.544618,0.455382,-0.029415,0.029415
2,2023090700,2023,1,09/07/2023,20:20:00,KC,DET,1940,(:09) (Shotgun) J.Goff pass incomplete deep ri...,2,...,COVER_2_ZONE,NaN,0,0,0.281891,-0.081964,0.771994,0.228006,0.000791,-0.000791
3,2023090700,2023,1,09/07/2023,20:20:00,KC,DET,1711,"(:45) (No Huddle, Shotgun) P.Mahomes pass deep...",2,...,COVER_2_ZONE,NaN,26,26,3.452352,2.342947,0.663187,0.336813,0.041843,-0.041843
4,2023090700,2023,1,09/07/2023,20:20:00,KC,DET,1588,(1:54) (Shotgun) P.Mahomes pass incomplete dee...,2,...,COVER_4_ZONE,NaN,0,0,1.921525,-0.324035,0.615035,0.384965,0.000061,-0.000061


,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,...,y,s,a,dir,o,num_frames_output,ball_land_x,ball_land_y,source_file,source_week
0,2023090700,101,False,54527,1,right,42,Bryan Cook,6-1,210,...,36.94,0.09,0.39,322.40,238.24,21,63.259998,-0.22,input_2023_w01.csv,1
1,2023090700,101,False,54527,2,right,42,Bryan Cook,6-1,210,...,36.94,0.04,0.61,200.89,236.05,21,63.259998,-0.22,input_2023_w01.csv,1
2,2023090700,101,False,54527,3,right,42,Bryan Cook,6-1,210,...,36.93,0.12,0.73,147.55,240.60,21,63.259998,-0.22,input_2023_w01.csv,1
3,2023090700,101,False,54527,4,right,42,Bryan Cook,6-1,210,...,36.92,0.23,0.81,131.40,244.25,21,63.259998,-0.22,input_2023_w01.csv,1
4,2023090700,101,False,54527,5,right,42,Bryan Cook,6-1,210,...,36.90,0.35,0.82,123.26,244.25,21,63.259998,-0.22,input_2023_w01.csv,1


,game_id,play_id,nfl_id,frame_id,x,y,source_file,source_week
0,2023090700,101,46137,1,56.22,17.28,output_2023_w01.csv,1
1,2023090700,101,46137,2,56.63,16.88,output_2023_w01.csv,1
2,2023090700,101,46137,3,57.06,16.46,output_2023_w01.csv,1
3,2023090700,101,46137,4,57.48,16.02,output_2023_w01.csv,1
4,2023090700,101,46137,5,57.91,15.56,output_2023_w01.csv,1


players per play


count    14108.000000
mean        12.273178
std          1.118459
min          1.000000
25%         12.000000
50%         13.000000
75%         13.000000
max         17.000000
Name: nfl_id, dtype: float64

...

count    14108.000000
mean         3.263751
std          1.312664
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          9.000000
Name: nfl_id, dtype: float64

input frames per play


count    14108.000000
mean        28.133967
std          9.257372
min          8.000000
25%         22.000000
50%         26.000000
75%         32.000000
max        123.000000
Name: frame_id, dtype: float64

output frames per play


count    14108.000000
mean        11.366601
std          5.147501
min          5.000000
25%          8.000000
50%         10.000000
75%         13.000000
max         94.000000
Name: frame_id, dtype: float64

## Split By Game

In [4]:
from ghostplayer.data.bdb2026 import split_game_ids_2026
from ghostplayer.utils.config import SplitConfig


split_config = SplitConfig(random_seed=7)
splits = split_game_ids_2026(tables.supplementary, split_config)

print(f"train games: {len(splits.train_game_ids)}")
print(f"validation games: {len(splits.validation_game_ids)}")
print(f"test games: {len(splits.test_game_ids)}")

train games: 244
validation games: 52
test games: 53


## Build And Save Graph Datasets

In [5]:
from dataclasses import asdict

from ghostplayer.data.bdb2026 import build_2026_graph_dataset, build_position_vocab_2026
from ghostplayer.data.build_graphs import load_graph_dataset, save_graph_dataset


position_vocab = build_position_vocab_2026(tables.input_tracking)
split_game_ids = {
    "train": splits.train_game_ids,
    "validation": splits.validation_game_ids,
    "test": splits.test_game_ids,
}

trajectory_horizon = int(tables.output_tracking["frame_id"].max()) if TRAJECTORY_HORIZON is None else int(TRAJECTORY_HORIZON)
print(f"trajectory_horizon: {trajectory_horizon} output frames")

summaries = {}
saved_paths = {}
for split_name, game_ids in split_game_ids.items():
    dataset, summary, position_vocab = build_2026_graph_dataset(
        tables.input_tracking,
        tables.output_tracking,
        game_ids=game_ids,
        lookback_frames=LOOKBACK_FRAMES,
        target_output_frame_id=TARGET_OUTPUT_FRAME_ID,
        trajectory_horizon=trajectory_horizon,
        position_vocab=position_vocab,
    )
    output_path = PROCESSED_DIR / f"{split_name}_graphs.npz"
    save_graph_dataset(dataset, output_path)
    summaries[split_name] = asdict(summary)
    saved_paths[split_name] = str(output_path)
    print(f"{split_name}: {summary.examples_built} examples -> {output_path}")
    print(f"  history_continuous: {dataset.history_continuous.shape}")
    print(f"  target_trajectories: {dataset.target_trajectories.shape}")
    print(f"  prediction nodes per play: min={dataset.defender_mask.sum(axis=1).min()}, max={dataset.defender_mask.sum(axis=1).max()}")

train: 9727 examples -> /Users/reezhan/Documents/GhostPlayer/processed/train_graphs.npz
  history_continuous: (9727, 10, 23, 6)
  prediction nodes per play: min=1, max=9
validation: 2187 examples -> /Users/reezhan/Documents/GhostPlayer/processed/validation_graphs.npz
  history_continuous: (2187, 10, 23, 6)
  prediction nodes per play: min=1, max=8
test: 2181 examples -> /Users/reezhan/Documents/GhostPlayer/processed/test_graphs.npz
  history_continuous: (2181, 10, 23, 6)
  prediction nodes per play: min=1, max=8


## Save Metadata

In [6]:
metadata = {
    "source": "nfl-big-data-bowl-2026-analytics",
    "data_root": str(data_root),
    "schema_mode": "2026-focused-player-subset",
    "lookback_frames": LOOKBACK_FRAMES,
    "target_output_frame_id": TARGET_OUTPUT_FRAME_ID,
    "trajectory_horizon": trajectory_horizon,
    "node_contract": "up to 22 provided player-context nodes plus one ball-landing context node",
    "prediction_mask": "stored in GraphDataset.defender_mask for compatibility; represents player_to_predict, not all defenders",
    "split_game_ids": {name: [int(game_id) for game_id in ids] for name, ids in split_game_ids.items()},
    "summaries": summaries,
    "position_vocab": position_vocab,
    "saved_paths": saved_paths,
}

metadata_path = PROCESSED_DIR / "dataset_metadata_2026.json"
metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True))
print(f"metadata -> {metadata_path}")

metadata -> /Users/reezhan/Documents/GhostPlayer/processed/dataset_metadata_2026.json


## Verify Saved Artifacts

In [7]:
for split_name, path_text in saved_paths.items():
    dataset = load_graph_dataset(Path(path_text))
    print(split_name)
    print(f"  edge_index:          {dataset.edge_index.shape}")
    print(f"  history_continuous:  {dataset.history_continuous.shape}")
    print(f"  target_positions:    {dataset.target_positions.shape}")
    print(f"  target_trajectories: {dataset.target_trajectories.shape if dataset.target_trajectories is not None else None}")
    print(f"  trajectory_mask:     {dataset.target_trajectory_mask.shape if dataset.target_trajectory_mask is not None else None}")
    print(f"  prediction_mask:     {dataset.defender_mask.shape}")
    print(f"  metadata:            {dataset.metadata.shape}")

train
  edge_index:          (2, 506)
  history_continuous:  (9727, 10, 23, 6)
  target_positions:    (9727, 23, 2)
  prediction_mask:     (9727, 23)
  metadata:            (9727, 4)
validation
  edge_index:          (2, 506)
  history_continuous:  (2187, 10, 23, 6)
  target_positions:    (2187, 23, 2)
  prediction_mask:     (2187, 23)
  metadata:            (2187, 4)
test
  edge_index:          (2, 506)
  history_continuous:  (2181, 10, 23, 6)
  target_positions:    (2181, 23, 2)
  prediction_mask:     (2181, 23)
  metadata:            (2181, 4)


## Optional: Train Baseline

In [8]:
import subprocess, sys
subprocess.check_call([
    sys.executable,
    "-m",
    "ghostplayer.training.train_baseline",
    "--train-data",
    str(PROCESSED_DIR / "train_graphs.npz"),
    "--validation-data",
    str(PROCESSED_DIR / "validation_graphs.npz"),
    "--output-dir",
    str(PROCESSED_DIR / "models"),
    "--epochs",
    "20",
])

epoch=1 train_loss=298.558939 train_ade=18.8213 validation_loss=12.799036 validation_ade=4.1571
epoch=2 train_loss=38.202436 train_ade=7.3952 validation_loss=6.138412 validation_ade=3.0012
epoch=3 train_loss=27.824194 train_ade=6.3186 validation_loss=4.409160 validation_ade=2.5791
epoch=4 train_loss=24.276059 train_ade=5.8689 validation_loss=3.619101 validation_ade=2.3297
epoch=5 train_loss=21.506371 train_ade=5.5176 validation_loss=4.694556 validation_ade=2.7693
epoch=6 train_loss=20.036983 train_ade=5.3048 validation_loss=6.712263 validation_ade=3.4191
epoch=7 train_loss=18.491293 train_ade=5.1079 validation_loss=3.440700 validation_ade=2.3346
epoch=8 train_loss=17.335393 train_ade=4.9288 validation_loss=4.455587 validation_ade=2.7362
epoch=9 train_loss=16.062185 train_ade=4.7490 validation_loss=7.876701 validation_ade=3.7419
epoch=10 train_loss=15.401345 train_ade=4.6530 validation_loss=4.924793 validation_ade=2.8537
epoch=11 train_loss=14.814742 train_ade=4.5543 validation_loss=9.1

0